In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [ ]:
%store -r combined_df
combined_df


In [ ]:
df = combined_df.reset_index(drop=True)

df = df.drop(columns=['sex','address','famsize','Pstatus','Fedu','Mjob','Fjob','guardian','schoolsup','famsup','paid','activities','romantic','goout'])

df

In [ ]:
X = df.drop(columns=["G3", "G2", "G1"])
y = df["G3"]


It is time to chose what is the best Model to fit the data, for better prediction accuracy.

In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
from sklearn.tree import DecisionTreeRegressor 
from sklearn.tree import plot_tree

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split ,cross_val_score

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OrdinalEncoder



In [ ]:
X.dtypes

In [ ]:
X_encoded = pd.get_dummies(X, columns=['school', 'reason', 'nursery', 'higher', 'internet'])
X_encoded.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded,y,random_state = 0)

rt = DecisionTreeRegressor(random_state = 0)
rt = rt.fit(X_train, y_train)

In [ ]:
plt.figure(figsize=(15,7.5))
plot_tree(rt,
         filled = True,
         rounded = True,
         feature_names = X_encoded.columns)
plt.show()

In [ ]:
path = rt.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
ccp_alphas = ccp_alphas[:-1]  

rts = []  


for alpha in ccp_alphas:
    rt = DecisionTreeRegressor(random_state=0, ccp_alpha=alpha)
    rt.fit(X_train, y_train)
    rts.append(rt)


In [ ]:
train_scores = [rt.score(X_train, y_train) for rt in rts]
test_scores = [rt.score(X_test, y_test) for rt in rts]

fig, ax= plt.subplots()
ax.set_xlabel("Alpha")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy Vs Alpha for training and test sets")
ax.plot(ccp_alphas, train_scores, marker = 'o',label = 'Train', drawstyle = 'steps-post')
ax.plot(ccp_alphas, test_scores, marker = 'o', label = 'Test', drawstyle = 'steps-post')
ax.legend()
plt.show()

The Decision tree as shown in the figure above, overfits the Training data, while it uderfits the testing data, but then they make it in a sweetspot where alpha is about 0.3

Using different testing and training data sets with the same alpha results in different accuracies , thats why it is fendamental to use cross validation 

In [ ]:
alpha_loop_values = []

for ccp_alpha in ccp_alphas:
    rt = DecisionTreeRegressor(random_state = 0, ccp_alpha = ccp_alpha)
    scores = cross_val_score(rt, X_train, y_train, cv= 5)
    alpha_loop_values.append([ccp_alpha, np.mean(scores),np.std(scores)])


alpha_results= pd.DataFrame(alpha_loop_values, columns= ['alpha','mean_accuracy','std'])

alpha_results.plot(x= 'alpha',
                  y = 'mean_accuracy',
                  yerr = 'std',
                  marker= 'o',
                  linestyle = '--')

In [ ]:
optimal_alpha= alpha_results[(alpha_results['alpha']> 0.29) 
    & 
    (alpha_results['alpha']< 0.33)] ['alpha']
optimal_alpha

In [ ]:
optimal_alpha = float(optimal_alpha)
optimal_alpha

In [ ]:
rt_pruned = DecisionTreeRegressor(random_state = 0, ccp_alpha= optimal_alpha)
rt_pruned = rt_pruned.fit(X_train, y_train)


In [ ]:

plt.figure(figsize=(15, 9), dpi=200)

plot_tree(
    rt_pruned,
    filled=True,
    rounded=True,
    feature_names=X_encoded.columns,
    max_depth=3,            
    fontsize=15,             
    precision=2,            
    proportion=True         
)

plt.tight_layout()
plt.show()
